## 03 - Data Anonymization

### Objective

This notebook prepares the extracted datasets for ethical analysis by removing personally identifiable information (PII).

Before generating anonymous student identifiers, the student records from the three academic terms are compared to verify that they represent the same cohort. This validation ensures that a single, consistent Student_ID can be assigned to each student and used across all datasets throughout the project.

The anonymized datasets produced in this notebook will be used for all subsequent data cleaning, analysis, SQL integration, and Power BI reporting.

In [1]:
import pandas as pd

### Loading the Student Datasets

The Student datasets extracted from the three academic term workbooks are loaded for comparison.

In [2]:
term1_students = pd.read_csv("../data/interim/term1/term1_student.csv")

term2_students = pd.read_csv("../data/interim/term2/term2_student.csv")

term3_students = pd.read_csv("../data/interim/term3/term3_student.csv")

### Inspecting the Student Datasets

The structure of each dataset is reviewed to confirm that the expected variables are available before comparing the student records.

In [3]:
print("Term 1")
display(term1_students.head())

print("Term 2")
display(term2_students.head())

print("Term 3")
display(term3_students.head())

Term 1


,STUDENT ID,STUDENT NAME
0,4056,AGBENOR ALFRED
1,567,AGBENU JORDAN
2,749,AGBO PRINCE
3,2050,AGBOTSE EBENEZER
4,696,APPIAH EMMANUEL


Term 2


,STUDENT ID,STUDENT NAME
0,4056,AGBENOR ALFRED
1,567,AGBENU JORDAN
2,749,AGBO PRINCE
3,2050,AGBOTSE EBENEZER
4,696,APPIAH EMMANUEL


Term 3


,STUDENT ID,STUDENT NAME
0,4056,AGBENOR ALFRED
1,567,AGBENU JORDAN
2,749,AGBO PRINCE
3,2050,AGBOTSE EBENEZER
4,696,APPIAH EMMANUEL


## Comparing the Number of Student Records

The number of student records in each academic term is compared to identify any differences in cohort size.

In [4]:
comparison = pd.DataFrame({
    "Term": ["Term 1", "Term 2", "Term 3"],
    "Students": [
        len(term1_students),
        len(term2_students),
        len(term3_students)
    ]
})

comparison

,Term,Students
0,Term 1,59
1,Term 2,58
2,Term 3,54


In [5]:
# Select only the columns required for comparison

term1_compare = term1_students[["STUDENT ID", "STUDENT NAME"]].copy()

term2_compare = term2_students[["STUDENT ID", "STUDENT NAME"]].copy()

term3_compare = term3_students[["STUDENT ID", "STUDENT NAME"]].copy()

In [6]:
for df in [term1_compare, term2_compare, term3_compare]:

    df["STUDENT NAME"] = (
        df["STUDENT NAME"]
        .str.strip()
        .str.upper()
    )

In [7]:
term1_vs_term2 = term1_compare.merge(
    term2_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term1_vs_term2

,STUDENT ID,STUDENT NAME,_merge
0,140,TETTEH MAVIS,both
1,160,GAMOR ESTHER,both
2,2050,AGBOTSE EBENEZER,both
3,2325,ASHANGBOR ABIGAIL,both
4,267,OFFEI JUSTICE,both
5,269,BONNEY CHARITY,both
6,270,DJABA BELINDA,both
7,271,ESHUN EVANS,both
8,272,ASIGBEY JOYCE,both
9,300,GOMASHIE PRINCE,both


In [8]:
term1_vs_term2[
    term1_vs_term2["_merge"] != "both"
]

,STUDENT ID,STUDENT NAME,_merge
49,727,AWUAH HEHELI JERRY,left_only
50,727,AWUAH KEKELI JERRY,right_only
53,744,OCQUAYE ANGELINA NAA SHORMEY,left_only


In [9]:
term1_vs_term3 = term1_compare.merge(
    term3_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term1_vs_term3[
    term1_vs_term3["_merge"] != "both"
]

,STUDENT ID,STUDENT NAME,_merge
5,269,BONNEY CHARITY,left_only
14,365,AGBENYEFIA BRIDGET,left_only
34,508,MENSAH VANESSA,left_only
49,727,AWUAH HEHELI JERRY,left_only
50,727,AWUAH KEKELI JERRY,right_only
53,744,OCQUAYE ANGELINA NAA SHORMEY,left_only
55,768,BRAHENE PRISCILLA,left_only


In [10]:
term2_vs_term3 = term2_compare.merge(
    term3_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term2_vs_term3[
    term2_vs_term3["_merge"] != "both"
]

,STUDENT ID,STUDENT NAME,_merge
5,269,BONNEY CHARITY,left_only
14,365,AGBENYEFIA BRIDGET,left_only
34,508,MENSAH VANESSA,left_only
53,768,BRAHENE PRISCILLA,left_only


### Investigating Student Record Differences

The comparison identified differences between the student records across the three academic terms.

These differences may reflect changes in enrollment, such as withdrawals, or new admissions. Alternatively, they may indicate inconsistencies in data entry.

The unmatched records will be reviewed before generating the master student list.

## Creating a Master Student List

The student records from all three academic terms are combined into a single master list.

This ensures that every student who appeared during the academic year is represented before anonymization.

In [11]:
master_students = pd.concat(
    [
        term1_compare,
        term2_compare,
        term3_compare
    ],
    ignore_index=True
)

master_students.head()

,STUDENT ID,STUDENT NAME
0,4056,AGBENOR ALFRED
1,567,AGBENU JORDAN
2,749,AGBO PRINCE
3,2050,AGBOTSE EBENEZER
4,696,APPIAH EMMANUEL


In [12]:
master_students = (
    master_students
    .drop_duplicates(subset="STUDENT ID")
    .sort_values("STUDENT ID")
    .reset_index(drop=True)
)

master_students

,STUDENT ID,STUDENT NAME
0,140,TETTEH MAVIS
1,160,GAMOR ESTHER
2,2050,AGBOTSE EBENEZER
3,2325,ASHANGBOR ABIGAIL
4,267,OFFEI JUSTICE
5,269,BONNEY CHARITY
6,270,DJABA BELINDA
7,271,ESHUN EVANS
8,272,ASIGBEY JOYCE
9,300,GOMASHIE PRINCE


In [13]:
print("Total unique students:", len(master_students))

Total unique students: 57


In [14]:
master_students = (
    master_students
    .sort_values("STUDENT ID")
    .reset_index(drop=True)
)

master_students["Student_ID"] = [
    f"ST{str(i).zfill(4)}"
    for i in range(1, len(master_students) + 1)
]

In [15]:
# Master students with anonymized IDs
master_students["Student_ID"] = [
    f"ST{str(i).zfill(4)}"
    for i in range(1, len(master_students) + 1)
]

master_students

,STUDENT ID,STUDENT NAME,Student_ID
0,140,TETTEH MAVIS,ST0001
1,160,GAMOR ESTHER,ST0002
2,2050,AGBOTSE EBENEZER,ST0003
3,2325,ASHANGBOR ABIGAIL,ST0004
4,267,OFFEI JUSTICE,ST0005
5,269,BONNEY CHARITY,ST0006
6,270,DJABA BELINDA,ST0007
7,271,ESHUN EVANS,ST0008
8,272,ASIGBEY JOYCE,ST0009
9,300,GOMASHIE PRINCE,ST0010


In [16]:
# Save the master student list to a CSV file
master_students.to_csv(
    "../data/metadata/student_lookup.csv",
    index=False
)

### Loading the Student Lookup Table

The student lookup table contains the mapping between the original school-assigned student identifiers and the anonymous Student_ID values thats was generated for this project.

This lookup table will be used to anonymize all extracted datasets while maintaining consistent relationships across the academic terms.

In [24]:
student_lookup = pd.read_csv("../data/metadata/student_lookup.csv")

student_lookup.head()

,STUDENT ID,STUDENT NAME,Student_ID
0,140,TETTEH MAVIS,ST0001
1,160,GAMOR ESTHER,ST0002
2,2050,AGBOTSE EBENEZER,ST0003
3,2325,ASHANGBOR ABIGAIL,ST0004
4,267,OFFEI JUSTICE,ST0005


### Creating a Reusable Anonymization Function

A reusable function is created to anonymize datasets containing student information.

The function performs the following tasks:

- Loads the dataset.
- Matches each student using the school-assigned STUDENT ID.
- Appends the anonymous Student_ID.
- Removes personally identifiable information.
- Saves the anonymized dataset for subsequent processing.

In [42]:
def anonymize_dataset(df, lookup):

    if "STUDENT ID" in df.columns:

        anonymized = df.merge(
            lookup[["STUDENT ID", "Student_ID"]],
            on="STUDENT ID",
            how="left"
        )

    elif "STUDENT NAME" in df.columns:

        anonymized = df.merge(
            lookup[["STUDENT NAME", "Student_ID"]],
            on="STUDENT NAME",
            how="left"
        )

    elif "NAME" in df.columns:

        lookup_name = lookup.rename(
            columns={"STUDENT NAME": "NAME"}
        )

        anonymized = df.merge(
            lookup_name[["NAME", "Student_ID"]],
            on="NAME",
            how="left"
        )

    else:

        raise ValueError("No student identifier found.")

    anonymized.insert(
        0,
        "Student_ID",
        anonymized.pop("Student_ID")
    )

    return anonymized

## Testing the Anonymization Function

The Attendance dataset is used to verify that the anonymization function correctly assigns Student_ID values before removing identifying information.

In [26]:
attendance = pd.read_csv("../data/interim/term1/term1_attendance.csv")

attendance.head()

,NAME,Attendance
0,AGBENOR ALFRED,69.0
1,AGBENU JORDAN,63.0
2,AGBO PRINCE,55.0
3,AGBOTSE EBENEZER,69.0
4,APPIAH EMMANUEL,54.0


In [39]:
attendance.columns.tolist()

['NAME', 'Attendance']

In [40]:
datasets = {
    "Student": term1_students,
    "Attendance": pd.read_csv("../data/interim/term1/term1_attendance.csv"),
    "Test 1": pd.read_csv("../data/interim/term1/term1_test_1.csv"),
    "Test 2": pd.read_csv("../data/interim/term1/term1_test_2.csv"),
    "Group Work": pd.read_csv("../data/interim/term1/term1_group_work.csv"),
    "Project Work": pd.read_csv("../data/interim/term1/term1_project_work.csv"),
    "Class Score": pd.read_csv("../data/interim/term1/term1_classscore.csv"),
    "Exam Score": pd.read_csv("../data/interim/term1/term1_examscore.csv"),
    "General Remarks": pd.read_csv("../data/interim/term1/term1_general_remarks.csv")
}

In [52]:
# Clean names in the attendance dataset
attendance["NAME"] = (
    attendance["NAME"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Clean names in the lookup table
student_lookup["STUDENT NAME"] = (
    student_lookup["STUDENT NAME"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [54]:
attendance["NAME"] = (
    attendance["NAME"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

student_lookup["STUDENT NAME"] = (
    student_lookup["STUDENT NAME"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

In [56]:
attendance_anonymized = attendance.merge(
    student_lookup.rename(columns={"STUDENT NAME": "NAME"})[
        ["NAME", "Student_ID"]
    ],
    on="NAME",
    how="left"
)

attendance_anonymized.head(56)

,NAME,Attendance,Student_ID
0,AGBENOR ALFRED,69.0,ST0019
1,AGBENU JORDAN,63.0,ST0037
2,AGBO PRINCE,55.0,ST0052
3,AGBOTSE EBENEZER,69.0,ST0003
4,APPIAH EMMANUEL,54.0,ST0044
5,ASARE RICHMOND,54.0,ST0027
6,ASIAMAH JOSEPH,65.0,ST0024
7,AWUAH HEHELI JERRY,69.0,ST0048
8,BAWA BASSIT,69.0,ST0016
9,DANQWAH PHILIP,69.0,ST0057


In [69]:
master_students = pd.concat(
    [
        term1_students[["STUDENT ID", "STUDENT NAME"]],
        term2_students[["STUDENT ID", "STUDENT NAME"]],
        term3_students[["STUDENT ID", "STUDENT NAME"]]
    ],
    ignore_index=True
)

master_students = (
    master_students
    .drop_duplicates(subset="STUDENT ID", keep="first")
    .reset_index(drop=True)
)

master_students.head()

,STUDENT ID,STUDENT NAME
0,4056,AGBENOR ALFRED
1,567,AGBENU JORDAN
2,749,AGBO PRINCE
3,2050,AGBOTSE EBENEZER
4,696,APPIAH EMMANUEL


In [73]:
master_students = pd.concat(
    [
        term1_students[["STUDENT ID", "STUDENT NAME"]],
        term2_students[["STUDENT ID", "STUDENT NAME"]],
        term3_students[["STUDENT ID", "STUDENT NAME"]]
    ],
    ignore_index=True
)

print(master_students.shape)

(171, 2)


In [74]:
master_students[
    master_students["STUDENT NAME"].str.contains(
        "SALIFU|HABLEAME",
        case=False,
        na=False
    )
]

,STUDENT ID,STUDENT NAME
45,385,HABLEAME SHILLA
52,437,SALIFU ZULAIHA
104,385,HABLEAME SHILLA
110,437,SALIFU ZULAIHA
160,385,HABLEAME SHILLA
165,437,SALIFU ZULAIHA
